# W5Q3 — Login Duration: Converters vs Non-Converters
**Question:** Does login duration on first visit predict trial conversion? Is there a statistically significant difference between converters and non-converters?

**Audience:** Product Team  
**Data source:** `ANALYTICS.MARTS.MART_WEB_BEHAVIOR`  
**SQL:** `sql/W5Q3_login_duration_converters_vs_not.sql`

---

> ⚠️ **Note:** The units of `login_duration` are not documented — could be seconds, minutes,
> or number of sessions. Inspect the distribution and flag the likely unit in findings.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W5Q3_login_duration_converters_vs_not.sql') as f:
    sql = f.read()

df = query(sql)
print(f"Rows: {len(df):,}")
print(f"Converters:     {df['converted_to_trial'].sum():,}")
print(f"Non-converters: {(~df['converted_to_trial']).sum():,}")
print(f"\nlogin_duration stats:")
display(df['login_duration'].describe())

## Distribution — Login Duration by Conversion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

converted     = df[df['converted_to_trial'] == True]['login_duration']
not_converted = df[df['converted_to_trial'] == False]['login_duration']

# Histogram
for data, label, color in [(converted, 'Converted', PALETTE[0]), (not_converted, 'Not Converted', PALETTE[1])]:
    axes[0].hist(data, bins=40, alpha=0.6, label=label, color=color, edgecolor='white', density=True)
axes[0].set_title('Login Duration Distribution', fontsize=12)
axes[0].set_xlabel('Login Duration')
axes[0].set_ylabel('Density')
axes[0].legend()
sns.despine(ax=axes[0])

# Box plot
df_plot = df[['login_duration', 'converted_to_trial']].copy()
df_plot['Group'] = df_plot['converted_to_trial'].map({True: 'Converted', False: 'Not Converted'})
sns.boxplot(data=df_plot, x='Group', y='login_duration', palette='muted', ax=axes[1], showfliers=False)
axes[1].set_title('Login Duration — Box Plot (outliers hidden)', fontsize=12)
axes[1].set_xlabel('')
axes[1].set_ylabel('Login Duration')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('../output/W5Q3_login_duration_distribution.png', bbox_inches='tight')
plt.show()

## Statistical Test — Mann-Whitney U

In [ ]:
stat, p = stats.mannwhitneyu(converted, not_converted, alternative='two-sided')
print(f"Mann-Whitney U statistic : {stat:,.0f}")
print(f"p-value                  : {p:.6f}")
print(f"Median — Converted       : {converted.median():.4f}")
print(f"Median — Not Converted   : {not_converted.median():.4f}")
print(f"Difference               : {converted.median() - not_converted.median():.4f}")
print()
if p < 0.05:
    print("✓ Statistically significant at α=0.05")
else:
    print("— Not statistically significant at α=0.05")

## Findings

- **Login duration units (inferred from distribution):** ...
- **Median login duration — converters:** ...
- **Median login duration — non-converters:** ...
- **Mann-Whitney U test result:** ...
- **Effect size / practical significance:** ...
- **Recommendation:** ...